# Colab — trening VegetationLSTM (NDVI/NDMI per piksel)

Pipeline:
1. Clone repo + instalacja
2. Montaż Drive z danymi (`fields_master`, `weather_daily`, TIFF-y lub gotowy `satellite_indices`)
3. Ekstrakcja **32 pikseli / pole** → `satellite_indices.parquet`
4. Trening LSTM na GPU
5. Predykcja dla wybranego `pixel_id` / `(lon, lat)`

**Runtime → Change runtime type → GPU**

## 0. Konfiguracja ścieżek

Dostosuj `DRIVE_DATA_DIR` do folderu na Google Drive, w którym leżą:
- `data/ml/fields_master.parquet`
- `data/ml/weather_daily.parquet`
- `data/fields/lowicki_fields_crops.geojson` (lub inny geojson pól z uprawami)
- `data/satellite/<tag>_<start>_to_<end>/` z plikami `.tif`

Albo wgraj już przebudowany `data/ml/satellite_indices.parquet` (ze schematem `pixel_id`) i ustaw `REBUILD_SATELLITE_INDICES = False`.

In [ ]:
REPO_URL = "https://github.com/kwroniak/field-under-pressure-satops.git"
REPO_DIR = "/content/field-under-pressure-satops"

# Folder na Drive z danymi (skopiowany lokalny `data/` lub jego podzbiór)
DRIVE_DATA_DIR = "/content/drive/MyDrive/fup_data/data"

FIELDS_GEOJSON = "data/fields/lowicki_fields_crops.geojson"
TIFF_DIR = "data/satellite/lowicki_2024-03-01_to_2024-06-30"
DATA_DIR = "data/ml"
CHECKPOINT = "ml/checkpoints/vegetation_lstm.pt"

PIXELS_PER_FIELD = 32
REBUILD_SATELLITE_INDICES = True  # False jeśli wgrywasz już gotowy parquet per-piksel

EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3

## 1. Clone + zależności

In [ ]:
import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo już jest: {REPO_DIR}")

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Drive → skopiuj `data/` do repo

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

src = Path(DRIVE_DATA_DIR)
dst = Path(REPO_DIR) / "data"
assert src.exists(), f"Brak folderu na Drive: {src}\nUstaw DRIVE_DATA_DIR w komórce konfiguracji."

dst.mkdir(parents=True, exist_ok=True)
# rsync-like: kopiuj zawartość data/ (nie nadpisuj całego repo)
!cp -a "{src}/." "{dst}/"

print("data/ml:", list((dst / "ml").glob("*.parquet")) if (dst / "ml").exists() else "BRAK")
print("TIFF-ów:", len(list(Path(TIFF_DIR).glob("*.tif")) + list(Path(TIFF_DIR).glob("*.tiff"))))

## 3. (Opcjonalnie) przebuduj `satellite_indices` — 32 px / pole

Usuwa stary plik ze średnią pola i buduje schemat:
`pixel_id, field_id, lon, lat, date, ndvi, ndmi`.

In [ ]:
from pathlib import Path
import pandas as pd
from ml.prepare_dataset import extract_satellite_indices_from_tiffs

out_sat = Path(DATA_DIR) / "satellite_indices.parquet"

if REBUILD_SATELLITE_INDICES:
    if out_sat.exists():
        out_sat.unlink()
        print("Usunięto stary satellite_indices.parquet")

    assert Path(FIELDS_GEOJSON).exists(), f"Brak {FIELDS_GEOJSON}"
    assert Path(TIFF_DIR).exists(), f"Brak {TIFF_DIR}"

    sat_df = extract_satellite_indices_from_tiffs(
        fields_geojson=FIELDS_GEOJSON,
        tiff_inputs=TIFF_DIR,
        out_path=out_sat,
        pixels_per_field=PIXELS_PER_FIELD,
    )
else:
    assert out_sat.exists(), f"Brak {out_sat} — wgraj plik albo ustaw REBUILD_SATELLITE_INDICES=True"
    sat_df = pd.read_parquet(out_sat)

print(sat_df.head(3))
print("rows:", len(sat_df), "| pixels:", sat_df["pixel_id"].nunique() if "pixel_id" in sat_df.columns else "BRAK pixel_id")
assert "pixel_id" in sat_df.columns, "To nadal stary format (średnia pola) — przebuduj indeksy."

## 4. Trening

In [ ]:
from ml.config import ModelConfig, TrainConfig
from ml.train import train_model

for name in ("fields_master.parquet", "weather_daily.parquet", "satellite_indices.parquet"):
    p = Path(DATA_DIR) / name
    assert p.exists(), f"Brak {p}"

result = train_model(
    data_dir=DATA_DIR,
    checkpoint_path=CHECKPOINT,
    model_cfg=ModelConfig(),
    train_cfg=TrainConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        split_mode="field",
        target_mode="observed",
    ),
)
print(result)

## 5. Predykcja — konkretny piksel

In [ ]:
import pandas as pd
from ml.predict import load_trained_model, predict_pixel_7days

fields_df = pd.read_parquet(Path(DATA_DIR) / "fields_master.parquet")
sat_df = pd.read_parquet(Path(DATA_DIR) / "satellite_indices.parquet")
weather_df = pd.read_parquet(Path(DATA_DIR) / "weather_daily.parquet")

model, scaler, m_cfg = load_trained_model(CHECKPOINT)

# Pierwszy dostępny piksel (albo wpisz własny pixel_id / lon+lat)
example_pixel = str(sat_df["pixel_id"].iloc[0])
field_id = str(sat_df.loc[sat_df["pixel_id"] == example_pixel, "field_id"].iloc[0])
row = fields_df[fields_df["field_id"].astype(str) == field_id].iloc[0]

pred = predict_pixel_7days(
    model=model,
    scaler=scaler,
    model_cfg=m_cfg,
    field_profile=row,
    satellite_df=sat_df,
    weather_df=weather_df,
    pixel_id=example_pixel,
    # lon=20.81, lat=52.61,  # alternatywa: najbliższy piksel z katalogu
)

print(f"pixel={pred['pixel_id']} field={pred['field_id']} lon={pred['lon']} lat={pred['lat']}")
print(pd.DataFrame({
    "date": pred["dates"],
    "ndvi": [pred["ndvi_by_date"][d] for d in pred["dates"]],
    "ndmi": [pred["ndmi_by_date"][d] for d in pred["dates"]],
}))

## 6. Zapisz checkpoint na Drive

In [ ]:
from pathlib import Path
import shutil

drive_out = Path("/content/drive/MyDrive/fup_data/checkpoints")
drive_out.mkdir(parents=True, exist_ok=True)
dst_ckpt = drive_out / Path(CHECKPOINT).name
shutil.copy2(CHECKPOINT, dst_ckpt)
print("Zapisano:", dst_ckpt)

# opcjonalnie: nowy satellite_indices też na Drive
sat_src = Path(DATA_DIR) / "satellite_indices.parquet"
sat_dst = Path("/content/drive/MyDrive/fup_data/data/ml/satellite_indices.parquet")
sat_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(sat_src, sat_dst)
print("Zapisano:", sat_dst)